### Data Cleaning Workbook

In [ ]:
%matplotlib inline

import os
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import matplotlib.lines as mlines
import matplotlib as mpl
import colorsys
import matplotlib.colors as mcolors

from eval_utils import load_model_results, lift_compression_args, include_peft, add_rank, add_dataset

from plot.color import Color
from plot.formatter import LabelFormatter, NumberFormatter, Scale, Grid, LambdaFormatter, Decorator, PercentFormatter

In [1]:
home = os.path.expanduser("~")
results_file = os.path.join(home, "src/compression-economics/compression_results.json")

print(f"Loading results from {results_file}")

x_col = "total_params"
y_col = "compression_factor"
save_path = "plot.png"  # None to disable saving

NameError: name 'os' is not defined

#### Loading and normalizing results

In [ ]:
print(f"Loading results from {results_file}")
dict_results = load_model_results(results_file)

# Lift compression args and flatten structure
dict_results = {k: lift_compression_args(v).get("compression", {}) for k, v in dict_results.items()}

# Include PEFT info and ranking
dict_results = include_peft(dict_results)
dict_results = add_rank(dict_results)
dict_results = add_dataset(dict_results)

print(f"Loaded {len(dict_results)} runs")

# Save normalized results for inspection
with open("loaded_results.json", "w") as f:
    json.dump(dict_results, f, indent=2)

print("Saved normalized results to loaded_results.json")

#TODO: handle quantized adapters

### Conversion to Dataframe 

In [ ]:
df = pd.DataFrame.from_dict(dict_results, orient="index").reset_index().rename(columns={"index": "run_key"})
df = df.fillna(0)

# drop Hf_token column from the results
df = df.drop(columns=["HF_token"], errors="ignore")

# Standardize encoding names
df["encoding"] = df["encoding"].replace({"huffman": "HC", "bitpacked": "BP"})

# Convert mb columns to bytes for consistency
MB = 1024 * 1024
df["adapter_size_bytes"] = (df["adapter_size_mb"] * MB).astype(int)
df["base_model_size_bytes"] = (df["base_model_size_mb"] * MB).astype(int)

# remove mb columns to avoid confusion, remove unnecessary columns for plotting
df = df.drop(columns=["adapter_size_mb", "base_model_size_mb", "print_results", 'output_path', 'lora_path'])

### Calculating more metrics

In [ ]:
# Fix datatypes of bytes by rounding up and converting to int
df["adapter_size_bytes"] = df["adapter_size_bytes"].astype(int)
df["base_model_size_bytes"] = df["base_model_size_bytes"].astype(int)
df["original_size_bytes"] = df["original_size_bytes"].astype(int)
df["arithmetic_code_size_bytes"] = df["arithmetic_code_size_bytes"].astype(int)
df["bitmap_size_bytes"] = df["bitmap_size_bytes"].astype(int)
df["final_size_bytes"] = df["final_size_bytes"].astype(int)

Helper functions

In [ ]:
# --- Full list of known models ---
ALL_MODELS = [
    "Qwen/Qwen2.5-0.5B",
    "Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-7B",
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-2B",
    "Qwen/Qwen3-4B",
    "gpt2",
    "google/flan-t5-small",
    "google/flan-t5-base",
    "google/flan-t5-large",
    "google/flan-t5-xl",
    "google/flan-t5-xxl",
    "meta-llama/Llama-3.2-1B-instruct",
    "state-spaces/mamba-130m-hf",
    "state-spaces/mamba-370m-hf",
    "state-spaces/mamba-790m-hf",
    "state-spaces/mamba-1.4b-hf",]
    
def clean_model_name(model):
    name = model.split("/")[-1]  # remove HF prefix
    name = name.replace("-hf", "")
    name = name.replace("-instruct", "")

    return name

def map_model_name(run_key):
    run_key_lower = run_key.lower()
    if "qwen2.5" in run_key_lower:
        model = next((m for m in ALL_MODELS if "qwen2.5" in m.lower()), "Unknown")
    elif "qwen3" in run_key_lower:
        model = next((m for m in ALL_MODELS if "qwen3" in m.lower()), "Unknown")
    elif "gpt2" in run_key_lower:
        model = "gpt2"
    elif "t5" in run_key_lower:
        model =  "t5"
    elif "mamba" in run_key_lower:
        model = next((m for m in ALL_MODELS if "mamba" in m.lower()), "Unknown")
    else:
        return "Unknown"
    
    return model 

df["model_name"] = df["run_key"].apply(map_model_name)

Safety Check

In [ ]:
print(df.columns)

print(df['model_name'].unique())
print(df['dataset'].unique())
print(df['input_path'].unique())
df.head(10)

In [ ]:
df[['model_name', 'rank', 'peft', 'original_size_bytes', 'arithmetic_code_size_bytes', 'bitmap_size_bytes', 'adapter_size_bytes', 'base_model_size_bytes', 'final_size_bytes']].head(20)